# Single Pair Simulation
This notebook loads settings and runs the pairs trading simulation for a single pair.

In [ ]:
import yaml
import sys
sys.path.append('..')

from src.strategies.bollinger import DynamicMeanReversionStrategy
from src.strategies.zscore import DynamicZScoreStrategy
from src.strategies.kalman import KalmanZScoreStrategy
from src.engine.backtest import PairsBacktest
from src.utils.metrics import evaluate_stationarity_regimes, apply_volatility_position_sizing
import datetime as dt
import pandas as pd
import numpy as np

# Load Settings
with open('../config/settings.yaml', 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)

# Dates setup
split_ratio = config['backtest']['split_ratio']
lookback_years = config['backtest']['lookback_years']
total_calendar_days = int(lookback_years * 365.25)

end_date = dt.datetime.strptime(config['backtest']['end_date'], '%Y-%m-%d').date()
start_date = end_date - pd.Timedelta(days=total_calendar_days)
train_days = int(total_calendar_days * split_ratio)
end_date_train = start_date + pd.Timedelta(days=train_days)
start_date_test = end_date_train + pd.Timedelta(days=1)

ticker_x = config['single_pair']['ticker_x']
ticker_y = config['single_pair']['ticker_y']
print(f"Running simulation for {ticker_y} vs {ticker_x}")


## 1. Bollinger Bands Strategy

In [ ]:
# Generate Grid from config
num_std_grid = [round(x, 2) for x in np.linspace(1.0, 2.0, 15)]
bollinger_param_grid = {
    'num_std': num_std_grid,
    'stop_mult': config['grids']['bollinger']['stop_mult'],
    'ma_window': config['grids']['bollinger']['ma_window']
}

backtest = PairsBacktest(
    ticker_x=ticker_x,
    ticker_y=ticker_y,
    start_date=str(start_date),
    end_date_train=str(end_date_train),
    start_date_test=str(start_date_test),
    end_date=str(end_date),
    strategy_cls=DynamicMeanReversionStrategy,
    strategy_param_grid=bollinger_param_grid,
    ols_windows_to_test=config['grids']['bollinger']['ols_windows'],
    show_heatmaps=True,
    transaction_cost=config['backtest']['transaction_cost'],
    compounding=config['backtest']['compounding'],
    target_metric=config['backtest']['target_metric'],
    strategy_name='Bollinger Bands'
)

final_train_df, final_test_df = backtest.execute_pipeline()
